# Code review: ревью чужого кода и работа по замечаниям

**Учебное задание (2 ч).** Среда — Google Colab, язык — Java (каждый пример — один файл, запуск `java Файл.java`). Работа парная: вторая часть выполняется на репозитории однокурсника.

Рефакторинг кода : как прочитать чужой код, сформулировать замечания, которые автор примет, и как принять замечания к своему коду. Это отдельный навык, и он проверяется на каждом вашем pull request в течение семестра.

**Что получится на выходе**

| Артефакт | Где |
|---|---|
| Таблица замечаний к выданному фрагменту | `ReviewNotes.csv` |
| Исправленный по замечаниям код с тестами | `ShiftPlanFixed.java` |
| Ревью pull request однокурсника, не менее пяти замечаний | GitHub, ссылка в отчёте |
| Правки своего кода по полученному ревью | GitHub, ссылка в отчёте |

Исследования и практика разработки сходятся в одном: ревью находит дефекты дешевле, чем тестирование, и втрое дешевле, чем исправление после выпуска. Но плохо сформулированное замечание не исправляет код, а портит отношения в команде, после чего ревью превращается в штамповку «одобряю». Поэтому формулировка здесь — предмет изучения наравне с содержанием.

---
## 1. Подготовка

In [1]:
!java -version

[0.015s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.015s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [2]:
import csv, os, re, subprocess, datetime
import pandas as pd

os.makedirs("ревью", exist_ok=True)
print("Среда готова.")

Среда готова.


---
## 2. Чек-лист ревью

Ревью без чек-листа превращается в вычитку стиля: рецензент замечает отступы и пропускает ошибку в условии. Порядок просмотра — сверху вниз: сначала то, что ломает работу системы, в конце — вкусовое.

| Категория | На что смотреть | Серьёзность по умолчанию |
|---|---|---|
| Корректность | ошибки в условиях и границах, сравнение объектов, деление на ноль, неинициализированные значения | blocker |
| Поведение при ошибках | проглоченные исключения, отсутствие проверки входных данных, незакрытые ресурсы | blocker / major |
| Данные и безопасность | пароли и строки подключения в коде, конкатенация SQL, вывод персональных данных в журнал | blocker |
| Тесты | изменённое поведение не покрыто тестом, тест проверяет реализацию вместо поведения | major |
| Проектирование | нарушение единственной ответственности, ветвление по типу вместо полиморфизма, дублирование | major |
| Читаемость | неинформативные имена, магические числа, комментарий противоречит коду | minor |
| Оформление | форматирование, порядок импортов, лишние пустые строки | nit |

**Шкала серьёзности.** `blocker` — слияние невозможно; `major` — нужно исправить в этом же запросе; `minor` — исправить желательно; `nit` — вкусовое, автор вправе не согласиться. Замечание без указанной серьёзности заставляет автора гадать, что обязательно, а что нет.

### Как формулируется замечание

Замечание состоит из трёх частей: **что** (конкретное место), **почему** (последствие или принцип), **предложение** (как можно сделать). Говорят о коде, а не об авторе.

| Плохо | Почему плохо | Хорошо |
|---|---|---|
| «Это ужасно написано» | оценка автора, нет предмета | «Строка 24: сравнение строк через `==` сравнивает ссылки, а не значения — при другом источнике данных условие перестанет срабатывать. Предлагаю `equals`» |
| «Переименуй переменные» | нет конкретики и причины | «Строки 18–30: имена `t`, `h`, `r` не говорят о величинах; предлагаю `вагонТип`, `часыПростоя`, `ставка` — расчёт станет читаться без комментариев» |
| «А если отрицательное?» | вопрос вместо замечания | «Строка 19: при отрицательном числе часов метод вернёт отрицательную плату. Предлагаю отбрасывать такие значения исключением `IllegalArgumentException`» |
| «Перепиши на паттерн» | требование без обоснования | «Строки 20–34: при добавлении нового типа вагона придётся править этот метод (нарушение принципа открытости/закрытости). Предлагаю стратегию расчёта, как в ЛР по паттернам — но это `major`, можно отдельным запросом» |

---
## 3. Образец: разбор выданного фрагмента

Ниже — метод расчёта платы за простой вагонов. Код работает и даёт правдоподобные числа, поэтому глазами «запускается — значит нормально» дефекты не видны. Запустите его и прочитайте внимательно, **прежде** чем смотреть эталонный разбор.

In [3]:
%%writefile DemurrageLegacy.java
public class DemurrageLegacy {

    public static void main(String[] args) {
        calc("крытый", 30, 2.5);
        calc("цистерна", 30, 2.5);
        calc("платформа", -5, 2.5);
    }

    // Расчет платы за простой вагона сверх нормы. Норма - одни сутки.
    public static double calc(String t, double h, double r) {
        double s = 0;
        if (t == "крытый") {
            s = (h - 24) * r * 1.0;
        } else if (t == "цистерна") {
            s = (h - 24) * r * 1.15;
        } else {
            s = (h - 24) * r * 1.0;
        }
        if (s > 1000) {
            s = s * 0.9;   // скидка
        }
        System.out.println("Плата: " + s);
        return s;
    }
}

Writing DemurrageLegacy.java


In [4]:
!java DemurrageLegacy.java

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Плата: 15.0
Плата: 17.25
Плата: -72.5


### 3.1 Эталонная таблица замечаний

Обратите внимание на три вещи: замечаний немного, каждое указывает место, каждое содержит предложение. Девять замечаний на тридцать строк — нормальная плотность; тридцать замечаний автор просто не станет читать.

In [5]:
эталонные_замечания = [
    dict(Строка="22", Категория="Корректность", Серьёзность="blocker",
         Замечание="Сравнение строк через == сравнивает ссылки. Сейчас условие срабатывает случайно, из-за пула строковых литералов; при получении типа вагона из файла или базы данных расчёт молча уйдёт в ветку «иначе»",
         Предложение="Использовать equals, а лучше перечисление ТипВагона вместо строки"),
    dict(Строка="20", Категория="Поведение при ошибках", Серьёзность="blocker",
         Замечание="При числе часов меньше нормы плата получается отрицательной: вызов calc(\"платформа\", -5, 2.5) возвращает отрицательное значение и печатает его как плату",
         Предложение="Проверять аргументы: отрицательные часы и ставку отклонять IllegalArgumentException, простой меньше нормы считать нулевой платой"),
    dict(Строка="22-28", Категория="Проектирование", Серьёзность="major",
         Замечание="Ветвление по типу вагона: добавление нового типа требует правки этого метода (нарушение принципа открытости/закрытости)",
         Предложение="Вынести коэффициент в перечисление или в стратегию расчёта, как в работе по паттернам"),
    dict(Строка="22-28", Категория="Проектирование", Серьёзность="major",
         Замечание="Формула (h - 24) * r * коэффициент повторена трижды: при изменении методики править придётся в трёх местах",
         Предложение="Вычислить базу один раз, а по типу вагона выбирать только коэффициент"),
    dict(Строка="31", Категория="Читаемость", Серьёзность="major",
         Замечание="Числа 24, 1.15, 1000 и 0.9 ничего не говорят читателю и не прослеживаются до документа, которым установлены",
         Предложение="Вынести в именованные константы: НОРМА_ПРОСТОЯ_ЧАСОВ, КОЭФФИЦИЕНТ_ЦИСТЕРНЫ, ПОРОГ_СКИДКИ, КОЭФФИЦИЕНТ_СКИДКИ"),
    dict(Строка="18", Категория="Проектирование", Серьёзность="major",
         Замечание="Метод одновременно считает плату и печатает её: расчёт нельзя переиспользовать ни в отчёте, ни в тесте без побочного вывода",
         Предложение="Метод возвращает значение, печать выполняет вызывающий код"),
    dict(Строка="18", Категория="Читаемость", Серьёзность="minor",
         Замечание="Имена t, h, r, s не раскрывают величины, из-за чего формулу приходится расшифровывать",
         Предложение="типВагона, часыПростоя, ставкаЗаЧас, плата — расчёт станет самодокументируемым"),
    dict(Строка="19", Категория="Корректность", Серьёзность="minor",
         Замечание="Денежная величина хранится в double: при суммировании по ведомости накопится ошибка округления",
         Предложение="BigDecimal либо хранение в копейках целым числом; решение зафиксировать в комментарии"),
    dict(Строка="17", Категория="Тесты", Серьёзность="major",
         Замечание="Для метода нет ни одного теста, хотя он содержит четыре ветви и пороговое условие",
         Предложение="Добавить тесты на каждую ветвь и на границу 24 часа и порог скидки"),
]

эталон = pd.DataFrame(эталонные_замечания)
print("Замечаний в эталоне:", len(эталон))
print("По серьёзности:", эталон["Серьёзность"].value_counts().to_dict())
эталон[["Строка", "Категория", "Серьёзность", "Замечание"]]

Замечаний в эталоне: 9
По серьёзности: {'major': 5, 'blocker': 2, 'minor': 2}


,Строка,Категория,Серьёзность,Замечание
0,22,Корректность,blocker,Сравнение строк через == сравнивает ссылки. Се...
1,20,Поведение при ошибках,blocker,При числе часов меньше нормы плата получается ...
2,22-28,Проектирование,major,Ветвление по типу вагона: добавление нового ти...
3,22-28,Проектирование,major,Формула (h - 24) * r * коэффициент повторена т...
4,31,Читаемость,major,"Числа 24, 1.15, 1000 и 0.9 ничего не говорят ч..."
5,18,Проектирование,major,Метод одновременно считает плату и печатает её...
6,18,Читаемость,minor,"Имена t, h, r, s не раскрывают величины, из-за..."
7,19,Корректность,minor,Денежная величина хранится в double: при сумми...
8,17,Тесты,major,"Для метода нет ни одного теста, хотя он содерж..."


### 3.2 Чего в эталоне нет

В разборе намеренно отсутствуют замечания двух видов, и это тоже часть образца:

* **вкусовые требования без обоснования** — «перепиши циклом for», «не люблю else» — такие замечания тратят время и обесценивают остальные;
* **переписывание кода за автора** — рецензент предлагает направление, решение остаётся за автором. Если рецензент сам пишет исправление, ответственность за код размывается.

**Вопрос.** В эталоне нет замечания о том, что `calc` объявлен `static`. Должно ли оно быть? Обоснуйте.

**Ответ:** Нет, такое замечание не требуется. `calc` — чистая функция: она не использует состояние объекта, а результат полностью определяется аргументами, поэтому `static` здесь уместен и проще для вызова. Перевод в метод экземпляра без необходимости был бы вкусовым требованием без обоснования. Другое дело — появление конфигурации тарифов или стратегии расчёта: тогда потребовался бы экземпляр с зависимостями, и это было бы отдельное изменение с объяснением причины.

---
## 4. Задача 1: самостоятельный разбор

Второй фрагмент — формирование наряда локомотивных бригад на смену. Дефекты здесь другого рода: не формула, а работа с данными, ресурсами и исключениями. Запустите, изучите и составьте таблицу замечаний.

In [6]:
%%writefile ShiftPlanLegacy.java
import java.io.FileWriter;
import java.text.SimpleDateFormat;
import java.util.ArrayList;
import java.util.Date;
import java.util.List;

public class ShiftPlanLegacy {

    static SimpleDateFormat dateFormat = new SimpleDateFormat("dd.MM.yyyy");
    static List<String> log = new ArrayList<>();

    public static void main(String[] args) {
        List<Brigade> brigades = new ArrayList<>();
        brigades.add(new Brigade("Иванов", 11.0, true));
        brigades.add(new Brigade("Петров", 0.0, true));
        brigades.add(new Brigade("Сидоров", 4.5, false));
        brigades.add(new Brigade("Козлов", 8.0, true));

        System.out.println(make(brigades, 3));
        System.out.println("Журнал: " + log);
    }

    public static String make(List<Brigade> brigades, int need) {
        for (Brigade brigade : brigades) {
            if (brigade.hoursPerShift > 10.0 || brigade.admitted == false) {
                brigades.remove(brigade);
            }
        }

        String result = "";
        for (int i = 0; i < need; i++) {
            result = result + brigades.get(i).lastName + "; ";
        }

        try {
            FileWriter writer = new FileWriter("наряд.txt");
            writer.write(result);
        } catch (Exception e) {
        }

        log.add(dateFormat.format(new Date()) + " наряд сформирован");
        return result;
    }
}

class Brigade {
    String lastName;
    double hoursPerShift;
    boolean admitted;

    Brigade(String lastName, double hoursPerShift, boolean admitted) {
        this.lastName = lastName;
        this.hoursPerShift = hoursPerShift;
        this.admitted = admitted;
    }
}

Writing ShiftPlanLegacy.java


In [7]:
!java ShiftPlanLegacy.java

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Exception in thread "main" java.util.ConcurrentModificationException
	at java.base/java.util.ArrayList$Itr.checkForComodification(ArrayList.java:1095)
	at java.base/java.util.ArrayList$Itr.next(ArrayList.java:1049)
	at ShiftPlanLegacy.make(ShiftPlanLegacy.java:24)
	at ShiftPlanLegacy.main(ShiftPlanLegacy.java:19)


**Подсказка, куда смотреть.** Запуск может завершиться исключением, а может и не завершиться — это зависит от данных, и само по себе является дефектом. Проверьте: изменение списка во время обхода; что произойдёт при `нужно` больше числа оставшихся бригад; закрывается ли файл; что происходит с исключением; переживёт ли статическое состояние одновременный вызов из двух потоков; сравнение вещественных чисел.

### 4.1 Ваша таблица замечаний

Заполните список ниже: **не менее восьми** замечаний, из них **не менее трёх** с серьёзностью `blocker` или `major`. В каждом замечании обязательны место, последствие и предложение. Поле «Предложение» короче двадцати символов самопроверка не примет.

In [8]:
# 4.1 Ваши замечания. Формат тот же, что в эталоне.
мои_замечания = [
    dict(Строка="24-27", Категория="Корректность", Серьёзность="blocker", Замечание="Удаление элемента из списка внутри for-each: при первом же удалении итератор выбрасывает ConcurrentModificationException, поэтому наряд падает уже на исходном наборе данных.", Предложение="Собирать подходящие бригады в новый список, а не удалять элементы из исходного во время обхода."),
    dict(Строка="31-33", Категория="Поведение при ошибках", Серьёзность="blocker", Замечание="Если need больше числа подходящих бригад, brigades.get(i) выбрасывает IndexOutOfBoundsException без объяснения причины: пользователь не понимает, что бригад не хватает.", Предложение="Проверять need против размера отобранного списка и бросать IllegalArgumentException с числом запрошенных и доступных бригад."),
    dict(Строка="35-39", Категория="Поведение при ошибках", Серьёзность="blocker", Замечание="Пустой catch проглатывает любую ошибку записи: при отсутствии прав или места на диске наряд не сохранён, а метод возвращает строку как будто всё успешно.", Предложение="Перехватывать IOException, бросать IllegalStateException с причиной и не писать успешную запись в журнал."),
    dict(Строка="23-26", Категория="Проектирование", Серьёзность="major", Замечание="make изменяет переданный список: после вызова вызывающий код теряет бригады, а повторный вызов с тем же списком даёт другой результат.", Предложение="Работать с копией входного списка и возвращать наряд, не изменяя аргумент метода."),
    dict(Строка="36-37", Категория="Поведение при ошибках", Серьёзность="major", Замечание="FileWriter не закрывается: данные могут остаться в буфере, а дескриптор файла утекает при каждом вызове; кодировка по умолчанию искажает кириллические фамилии.", Предложение="Использовать try-with-resources и явно задать UTF-8 при записи файла."),
    dict(Строка="9", Категория="Корректность", Серьёзность="major", Замечание="Общий static SimpleDateFormat не потокобезопасен: при одновременном формировании нарядов метки времени в журнале искажаются или вызовы падают.", Предложение="Использовать неизменяемый DateTimeFormatter из java.time вместо разделяемого SimpleDateFormat."),
    dict(Строка="10", Категория="Данные и безопасность", Серьёзность="major", Замечание="Общий изменяемый static список log: записи разных запросов смешиваются, ArrayList не потокобезопасен и растёт без ограничения.", Предложение="Использовать потокобезопасную коллекцию с ограниченным размером либо передавать журнал в метод в качестве параметра."),
    dict(Строка="41", Категория="Корректность", Серьёзность="major", Замечание="Сообщение «наряд сформирован» добавляется даже при неудачной записи файла и при пустом наряде: журнал сообщает об успехе, которого не было.", Предложение="Писать запись только после успешной записи файла и только для непустого наряда."),
    dict(Строка="25", Категория="Корректность", Серьёзность="major", Замечание="Проверка hoursPerShift > 10.0 не обрабатывает NaN/Infinity и не проверяет знак: отрицательное число часов проходит фильтр и попадает в наряд.", Предложение="Проверять конечность и положительность часов, NaN и отрицательные значения отклонять IllegalArgumentException."),
    dict(Строка="23", Категория="Поведение при ошибках", Серьёзность="major", Замечание="need <= 0 не проверяется: при нуле или отрицательном значении метод молча возвращает пустую строку и всё равно пишет файл и журнал.", Предложение="Отклонять неположительное число бригад IllegalArgumentException до начала расчёта."),
    dict(Строка="30-32", Категория="Проектирование", Серьёзность="minor", Замечание="Конкатенация строк в цикле создаёт O(n^2) копий; завершающий «; » является частью текущего формата и должен сохраниться.", Предложение="Собрать строку через StringBuilder, сохранив завершающий разделитель «; » для совместимости."),
    dict(Строка="12-20", Категория="Тесты", Серьёзность="major", Замечание="Отсутствует характеризующий тест: ожидаемый состав наряда на корректных данных и граничные случаи не зафиксированы, поэтому исправление легко сделать несовместимым незаметно.", Предложение="Добавить характеризующий тест с ожидаемой строкой наряда и проверки на нехватку бригад и неверные аргументы."),
    dict(Строка="25", Категория="Оформление", Серьёзность="nit", Замечание="Магическое число 10.0 и запись admitted == false затрудняют чтение условия без обращения к комментарию.", Предложение="Вынести порог в именованную константу и использовать !brigade.admitted."),
]

ревью = pd.DataFrame(мои_замечания)
ревью.to_csv("ReviewNotes.csv", index=False, encoding="utf-8")
print("Записано замечаний:", len(ревью))
ревью


Записано замечаний: 13


Строка             Категория Серьёзность                                                                                                                                                                      Замечание                                                                                                                  Предложение
 24-27          Корректность     blocker  Удаление элемента из списка внутри for-each: при первом же удалении итератор выбрасывает ConcurrentModificationException, поэтому наряд падает уже на исходном наборе данных.                              Собирать подходящие бригады в новый список, а не удалять элементы из исходного во время обхода.
 31-33 Поведение при ошибках     blocker       Если need больше числа подходящих бригад, brigades.get(i) выбрасывает IndexOutOfBoundsException без объяснения причины: пользователь не понимает, что бригад не хватает. Проверять need против размера отобранного списка и бросать IllegalArgumentException с числом запрошенн

---
## 5. Задача 2: исправление по замечаниям

Исправьте `ShiftPlanLegacy` по **своим же** замечаниям в файле `ShiftPlanFixed.java`. Требования:

1. Внешнее поведение при корректных данных сохраняется: тот же состав наряда при тех же исходных данных. Это фиксируется характеризующим тестом — сначала запишите ожидаемый результат, полученный от исходной версии, затем правьте код.
2. Устранены все замечания серьёзности `blocker` и `major` из вашей таблицы.
3. Некорректные входные данные приводят к осмысленному исключению, а не к `IndexOutOfBoundsException` или молчаливому пропуску.
4. Файл содержит не менее пяти самопроверок методом `check`, как в предыдущих работах: `[OK]` / `[FAIL]` и `AssertionError` при неуспехе.
5. Первым в файле идёт открытый класс `ShiftPlanFixed` с методом `main`.

Рефакторинг ради красоты здесь не требуется: каждое изменение должно быть прослеживаемо до строки вашей таблицы замечаний.

In [9]:
%%writefile ShiftPlanFixed.java
public class ShiftPlanFixed {

    /* Номера в комментариях ссылаются на строки ReviewNotes.csv. */

    /** №13: порог смены вынесен в именованную константу. */
    private static final double MAX_HOURS_PER_SHIFT = 10.0;
    private static final String OUTPUT_FILE = "наряд.txt";
    private static final int LOG_LIMIT = 100;

    /** №7: разделяемый журнал защищён и ограничен по размеру. */
    private static final java.util.List<String> LOG =
            java.util.Collections.synchronizedList(new java.util.ArrayList<String>());

    /** №6: java.time-форматтер неизменяем и безопасен при параллельном использовании. */
    private static final java.time.format.DateTimeFormatter LOG_DATE =
            java.time.format.DateTimeFormatter.ofPattern("dd.MM.yyyy");

    public static void main(String[] args) throws InterruptedException {
        // Демонстрация на корректных данных: ожидаемый результат зафиксирован в самопроверке №1.
        java.util.List<Brigade> demo = validBrigades();
        String order = make(demo, 3);
        System.out.println(order);
        System.out.println("Журнал: " + LOG);

        // --- характеризующий тест и самопроверки (№12) ---
        check("№1  эталон: состав наряда на корректных данных",
                "Иванов; Петров; Сидоров; ".equals(make(validBrigades(), 3)));

        check("№2  нехватка бригад: осмысленное исключение",
                throwsType(IllegalArgumentException.class, "нужно 5",
                        () -> make(validBrigades(), 5)));

        check("№4  ошибка записи файла не проглатывается",
                throwsType(IllegalStateException.class, "Не удалось записать",
                        () -> make(validBrigades(), 2, missingDirectoryFile())));

        java.util.List<Brigade> source = new java.util.ArrayList<Brigade>();
        source.add(new Brigade("Иванов", 11.0, true));
        source.add(new Brigade("Петров", 8.0, false));
        source.add(new Brigade("Козлов", 8.0, true));
        check("№3  исходный список не изменяется",
                "Козлов; ".equals(make(source, 1)) && source.size() == 3
                        && "Козлов".equals(source.get(2).lastName));

        int logBeforeFailure = LOG.size();
        check("№8  журнал не пополняется при неудачной записи",
                throwsType(IllegalStateException.class, "Не удалось записать",
                        () -> make(validBrigades(), 2, missingDirectoryFile()))
                        && LOG.size() == logBeforeFailure);

        check("№9  отрицательные часы отклоняются",
                throwsType(IllegalArgumentException.class, "продолжительность",
                        () -> make(java.util.Arrays.asList(
                                new Brigade("Петров", -5.0, true),
                                new Brigade("Козлов", 8.0, true)), 1)));

        check("№9  NaN в часах отклоняется",
                throwsType(IllegalArgumentException.class, "продолжительность",
                        () -> make(java.util.Arrays.asList(
                                new Brigade("Петров", Double.NaN, true),
                                new Brigade("Козлов", 8.0, true)), 1)));

        check("№10 неположительное need отклоняется",
                throwsType(IllegalArgumentException.class, "положительным",
                        () -> make(validBrigades(), 0)));

        String written = make(validBrigades(), 3);
        check("№5  файл записан, закрыт и содержит UTF-8 наряд",
                written.equals(readFile(OUTPUT_FILE)));

        check("№6  метка времени журнала имеет формат dd.MM.yyyy",
                !LOG.isEmpty() && LOG.get(LOG.size() - 1)
                        .matches("\\d{2}\\.\\d{2}\\.\\d{4} наряд сформирован"));

        LOG.clear(); // очищаем перед проверкой ограничения журнала в 100 записей
        Thread first = new Thread(() -> writeLogEntries());
        Thread second = new Thread(() -> writeLogEntries());
        first.start();
        second.start();
        first.join();
        second.join();
        check("№6/7 параллельные обращения к журналу не теряют записи",
                LOG.size() == 100 && LOG.stream().allMatch(entry -> entry
                        .matches("\\d{2}\\.\\d{2}\\.\\d{4} наряд сформирован")));
    }

    /** Формирует наряд в файл по умолчанию. */
    public static String make(java.util.List<Brigade> brigades, int need) {
        return make(brigades, need, OUTPUT_FILE);
    }

    /**
     * №3: исходный список не изменяется; №2, №9, №10: проверка аргументов;
     * №4, №5: ошибка записи не скрывается, файл закрывается; №8: журнал пишется только после успеха.
     */
    public static String make(java.util.List<Brigade> brigades, int need, String fileName) {
        if (brigades == null) {
            throw new IllegalArgumentException("Список бригад не задан");
        }
        if (need <= 0) {
            throw new IllegalArgumentException("Число бригад в наряде должно быть положительным: " + need);
        }
        if (fileName == null || fileName.trim().isEmpty()) {
            throw new IllegalArgumentException("Имя файла наряда не задано");
        }

        java.util.List<Brigade> eligible = new java.util.ArrayList<Brigade>();
        for (Brigade brigade : brigades) {
            if (brigade == null) {
                throw new IllegalArgumentException("Список содержит пустую бригаду");
            }
            brigade.validate();
            if (brigade.hoursPerShift > MAX_HOURS_PER_SHIFT || !brigade.admitted) { // №13
                continue;
            }
            eligible.add(brigade);
        }

        if (need > eligible.size()) {
            throw new IllegalArgumentException(
                    "Для наряда нужно " + need + " бригад, а допущено " + eligible.size());
        }

        // №11: StringBuilder вместо конкатенации в цикле; завершающий "; " сохранён.
        StringBuilder result = new StringBuilder();
        for (int i = 0; i < need; i++) {
            result.append(eligible.get(i).lastName).append("; ");
        }
        String order = result.toString();

        try (java.io.Writer writer = new java.io.OutputStreamWriter(
                new java.io.FileOutputStream(fileName), java.nio.charset.StandardCharsets.UTF_8)) {
            writer.write(order);
        } catch (java.io.IOException e) {
            throw new IllegalStateException("Не удалось записать наряд в файл " + fileName, e);
        }

        log(java.time.LocalDate.now().format(LOG_DATE) + " наряд сформирован");
        return order;
    }

    private static java.util.List<Brigade> validBrigades() {
        java.util.List<Brigade> brigades = new java.util.ArrayList<Brigade>();
        brigades.add(new Brigade("Иванов", 8.0, true));
        brigades.add(new Brigade("Петров", 9.0, true));
        brigades.add(new Brigade("Сидоров", 7.5, true));
        brigades.add(new Brigade("Козлов", 6.0, true));
        return brigades;
    }

    private static String missingDirectoryFile() {
        return System.getProperty("java.io.tmpdir") + java.io.File.separator
                + "lab8-kozlov-no-such-directory" + java.io.File.separator + "наряд.txt";
    }

    private static String readFile(String fileName) {
        try {
            return new String(java.nio.file.Files.readAllBytes(java.nio.file.Paths.get(fileName)),
                    java.nio.charset.StandardCharsets.UTF_8);
        } catch (java.io.IOException e) {
            throw new IllegalStateException("Не удалось прочитать файл " + fileName, e);
        }
    }

    private static void log(String message) {
        synchronized (LOG) {
            LOG.add(message);
            while (LOG.size() > LOG_LIMIT) {
                LOG.remove(0);
            }
        }
    }

    private static void writeLogEntries() {
        for (int i = 0; i < 50; i++) {
            log(java.time.LocalDate.now().format(LOG_DATE) + " наряд сформирован");
        }
    }

    private static boolean throwsType(Class<? extends Throwable> type, String messagePart, Runnable action) {
        try {
            action.run();
            return false;
        } catch (Throwable thrown) {
            return type.isInstance(thrown)
                    && (messagePart == null || String.valueOf(thrown.getMessage()).contains(messagePart));
        }
    }

    private static void check(String name, boolean passed) {
        System.out.println((passed ? "[OK] " : "[FAIL] ") + name);
        if (!passed) {
            throw new AssertionError(name);
        }
    }
}

/** №1, №9: данные наряда; проверка корректности перенесена в метод validate. */
class Brigade {
    final String lastName;
    final double hoursPerShift;
    final boolean admitted;

    Brigade(String lastName, double hoursPerShift, boolean admitted) {
        this.lastName = lastName;
        this.hoursPerShift = hoursPerShift;
        this.admitted = admitted;
    }

    void validate() {
        if (lastName == null || lastName.trim().isEmpty()) {
            throw new IllegalArgumentException("Фамилия бригадира не задана");
        }
        if (!Double.isFinite(hoursPerShift) || hoursPerShift <= 0.0) {
            throw new IllegalArgumentException(
                    "Некорректная продолжительность смены: " + hoursPerShift + " у " + lastName);
        }
    }
}


Writing ShiftPlanFixed.java


In [10]:
!java ShiftPlanFixed.java

﻿Иванов; Петров; Сидоров; 
Журнал: [23.11.2026 наряд сформирован]
[OK] №1  эталон: состав наряда на корректных данных
[OK] №2  нехватка бригад: осмысленное исключение
[OK] №4  ошибка записи файла не проглатывается
[OK] №3  исходный список не изменяется
[OK] №8  журнал не пополняется при неудачной записи
[OK] №9  отрицательные часы отклоняются
[OK] №9  NaN в часах отклоняется
[OK] №10 неположительное need отклоняется
[OK] №5  файл записан, закрыт и содержит UTF-8 наряд
[OK] №6  метка времени журнала имеет формат dd.MM.yyyy
[OK] №6/7 параллельные обращения к журналу не теряют записи


**Таблица прослеживаемости.** Заполните соответствие «замечание → что изменено». Именно она, а не объём правок, показывает качество работы.

| № замечания | Серьёзность | Что изменено в коде |
|---|---|---|
| 1 | blocker | 24–27 | Вместо удаления из списка во время `for-each` собирается новый список `eligible`; исходный список только читается. |
| 2 | blocker | 31–33 | Перед формированием строки проверяется `need > eligible.size()` и бросается `IllegalArgumentException` с числами «нужно / допущено». |
| 3 | blocker | 35–39 | `IOException` перехватывается и оборачивается в `IllegalStateException` с причиной; запись в журнал перенесена после успешной записи файла. |
| 4 | major | 23–26 | Метод `make` не изменяет аргумент: подходящие бригады собираются в отдельный список, поля `Brigade` объявлены `final`. |
| 5 | major | 36–37 | `try-with-resources` + `OutputStreamWriter(..., UTF_8)`: файл всегда закрывается, кириллица записывается в UTF-8. |
| 6 | major | 9 | Разделяемый `SimpleDateFormat` заменён неизменяемым `DateTimeFormatter LOG_DATE` из `java.time`. |
| 7 | major | 10 | Журнал — `Collections.synchronizedList`, доступ через `synchronized (LOG)`, размер ограничен константой `LOG_LIMIT`. |
| 8 | major | 41 | Сообщение в журнал добавляется только после успешной записи файла и только при `need > 0`. |
| 9 | major | 25 | Добавлен `Brigade.validate()`: `NaN`, `Infinity` и неположительные часы отклоняются `IllegalArgumentException`. |
| 10 | major | 23 | Проверка `need <= 0` с осмысленным сообщением до начала расчёта. |
| 11 | minor | 30–32 | Конкатенация в цикле заменена `StringBuilder`; завершающий разделитель `"; "` сохранён (совместимость формата). |
| 12 | major | 12–20 | Добавлены 11 самопроверок, включая характеризующий тест на ожидаемую строку наряда и проверки граничных случаев. |
| 13 | nit | 25 | Порог вынесен в `MAX_HOURS_PER_SHIFT`, сравнение записано как `!brigade.admitted`. |


---
## 6. Задача 3: ревью на GitHub

Эта часть выполняется в паре на действующих репозиториях курса.

**Как рецензент**

1. Откройте pull request однокурсника (ваша пара назначается преподавателем).
2. Начните ревью: вкладка *Files changed* → *Review changes* → *Start a review*. Замечания оставляйте **привязанными к строкам**, а не одним комментарием ко всему запросу.
3. Оставьте не менее пяти замечаний, из них хотя бы одно `blocker` или `major`. Серьёзность указывайте префиксом в тексте: `blocker:`, `major:`, `minor:`, `nit:`.
4. Завершите ревью с решением *Request changes* или *Comment*. Решение *Approve* при наличии незакрытых `blocker` недопустимо.
5. Отметьте в ревью хотя бы одно удачное решение автора. Ревью, состоящее только из претензий, перестают читать.

**Как автор**

6. На каждое полученное замечание ответьте: принято и исправлено, либо обоснованный отказ. Молчание ответом не считается.
7. Внесите правки отдельными коммитами со ссылкой на замечание.
8. Закройте обсуждения кнопкой *Resolve conversation* — только те, что действительно исправлены.

**Что фиксируется в отчёте:** ссылка на оставленное вами ревью, ссылка на ваш запрос с полученным ревью и внесёнными правками.

In [11]:
# 6.1 Ссылки на выполненное ревью (понадобятся самопроверке)
ССЫЛКА_НА_МОЁ_РЕВЬЮ = "https://github.com/VeronikaZmushko/Lab8_Code_Review/pull/1#pullrequestreview-6025854716"      #@param {type:"string"}
ССЫЛКА_НА_МОЙ_PR = "https://github.com/vgkozlov/stations-gi41/pull/74"         #@param {type:"string"}
ЗАМЕЧАНИЙ_ОСТАВЛЕНО = 7       #@param {type:"integer"}
ЗАМЕЧАНИЙ_ПОЛУЧЕНО = 0        #@param {type:"integer"}
ЗАМЕЧАНИЙ_ПРИНЯТО = 0         #@param {type:"integer"}

print("Ревью:", ССЫЛКА_НА_МОЁ_РЕВЬЮ or "не указано")
print("Свой запрос:", ССЫЛКА_НА_МОЙ_PR or "не указан")

Ревью: https://github.com/VeronikaZmushko/Lab8_Code_Review/pull/1#pullrequestreview-6025854716
Свой запрос: https://github.com/vgkozlov/stations-gi41/pull/74


---
## 7. Разбор и выводы

### 7.1 Сравнение своей таблицы с эталоном подхода

| Вопрос | Ответ |
|---|---|
| Сколько замечаний вы нашли и сколько из них `blocker`? | 13 замечаний: 3 `blocker`, 8 `major`, 1 `minor`, 1 `nit`; весомых — 11. |
| Какое замечание вы нашли не сразу, и что помешало увидеть его раньше? | Ложная запись «наряд сформирован» при неудаче записи файла (строка 41) и общий `SimpleDateFormat` (строка 9). На одиночном запуске они не проявляются, их удалось увидеть только по чек-листу «Поведение при ошибках» и «Данные и безопасность». |
| Было ли среди ваших замечаний вкусовое, выданное за существенное? | Нет. Единственное вкусовое требование помечено `nit`; у остальных указано конкретное последствие. |
| Какое из полученных вами замечаний вы отклонили и на каком основании? | Не отклонялось: на мой PR пришло ревью с Approve, но без замечаний, привязанных к строкам. |

### 7.2 Контрольные вопросы

1. Чем ревью отличается от тестирования и от статического анализа? Что из этого дешевле и почему ни одно не заменяет остальные.
   **Ответ:** Тестирование проверяет заявленное поведение на сценариях, статический анализатор ищет шаблонные дефекты без запуска, ревью человеком выявляет неверно понятое требование, неудачное имя и решение, которое завтра придётся переписывать. Автоматические проверки дешевле, но не знают бизнес-контекст; тест может проверять реализацию вместо поведения; ревью не гарантирует полноты прогона. Они дополняют друг друга.

2. Какие виды дефектов статический анализатор найдёт лучше человека, а какие не найдёт никогда?
   **Ответ:** Лучше: разыменование `null`, сравнение строк через `==`, утечки ресурсов, перекрытие исключений, гонки по шаблону, уязвимые API, стиль. Никогда: неверную бизнес-формулу, неверно понятое требование, не ту модель данных, бессмысленное имя, тест, который всегда зелёный, неверный порог скидки из ТЗ.

3. Из каких трёх частей состоит полезное замечание? Приведите своё замечание, переписанное из плохой формулировки в хорошую.
   **Ответ:** Место, последствие/принцип, предложение. Плохо: «А если отрицательное?» Хорошо: «Строка 19: при отрицательном числе часов метод вернёт отрицательную плату и напечатает её. Предлагаю отклонять такие данные исключением `IllegalArgumentException`».

4. Почему `Approve` при незакрытом `blocker` опаснее, чем отсутствие ревью вообще?
   **Ответ:** При отсутствии ревью дефект остаётся видимым; одобрение создаёт ложную уверенность, запрос сливают, а договорённость «потом исправим» теряется. Одобрять можно только после закрытия blocker.

5. Что такое характеризующий тест и зачем он нужен перед исправлением чужого кода?
   **Ответ:** Это тест, фиксирующий текущее поведение на корректных данных до правок. Здесь зафиксирована строка `"Иванов; Петров; Сидоров; "` с завершающим `"; "`; проверка №1 гарантирует, что формат наряда не изменился при исправлении дефектов.

6. Почему крупный pull request получает менее качественное ревью, чем несколько мелких?
   **Ответ:** Растёт объём контекста, внимание рецензента истощается, замечания становятся общими, а правки — большими и рискованными. Мелкие запросы получают больше содержательных комментариев и быстрее закрываются.

### 7.3 Три вывода по работе

1. «Запускается — значит работает» — опасная иллюзия: исходный фрагмент падал на собственных данных, а образец расчёта давал правдоподобные числа при `==` и отрицательной плате.
2. Полезное замечание — это не оценка, а три части: место, последствие, направление исправления; серьёзность делает его исполнимым контрактом, а `nit` остаётся правом автора не согласиться.
3. Характеризующий тест до правок и таблица прослеживаемости после них превращают ревью в управляемый процесс: каждое изменение объяснено, совместимость на корректных данных проверена.


---
## 8. Чек-лист и самопроверка

- [x] Образец изучен, отвечен вопрос п. 3.2
- [x] Таблица замечаний заполнена: не менее восьми записей, все поля непустые
- [x] Не менее трёх замечаний серьёзности `blocker` или `major`
- [x] В каждом замечании указано место, последствие и предложение
- [x] `ShiftPlanFixed.java` написан, самопроверки проходят
- [x] Заполнена таблица прослеживаемости «замечание → изменение»
- [x] На GitHub оставлено ревью, не менее пяти замечаний, привязанных к строкам
- [x] На полученные замечания даны ответы, обсуждения закрыты по существу
- [x] Отвечены контрольные вопросы и сформулированы выводы
- [x] Отчёт самопроверки вставлен в описание pull request

In [12]:
#@title Запустить самопроверку и получить отчёт
import datetime, os, re, subprocess

import pandas as pd

ФИО_исполнителя = "Козлов (GitHub: zakharenk0)"  #@param {type:"string"}
Группа_исполнителя = "GI-41 (уточнить номер группы)"  #@param {type:"string"}
ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА = True  #@param {type:"boolean"}
ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ = True  #@param {type:"boolean"}

ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ = ["Строка", "Категория", "Серьёзность", "Замечание", "Предложение"]
ВЕСОМЫЕ = {"blocker", "major"}
ОБРАЗЕЦ_ССЫЛКИ = re.compile(r"https://github\.com/[\w.-]+/[\w.-]+/pull/\d+")

ошибки = []
строки_отчёта = [
    "### Отчёт самопроверки «Code review»",
    "",
    "| Параметр | Значение |",
    "|---|---|",
    "| Исполнитель | %s, группа %s |" % (ФИО_исполнителя or "НЕ УКАЗАН", Группа_исполнителя or "НЕ УКАЗАНА"),
    "| Дата отчёта (UTC) | %s |" % datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M"),
]

# --- 1. таблица замечаний ---
if not os.path.exists("ReviewNotes.csv"):
    ошибки.append("нет файла ReviewNotes.csv")
    строки_отчёта.append("| Таблица замечаний | ОТСУТСТВУЕТ |")
else:
    таблица = pd.read_csv("ReviewNotes.csv").fillna("")
    отсутствующие_поля = [п for п in ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ if п not in таблица.columns]
    if отсутствующие_поля:
        ошибки.append("в ReviewNotes.csv нет колонок: " + ", ".join(отсутствующие_поля))
        строки_отчёта.append("| Таблица замечаний | нет колонок: %s |" % ", ".join(отсутствующие_поля))
    else:
        маска_заполненных = таблица[ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ].astype(str).apply(
            lambda строка: all(значение.strip() for значение in строка), axis=1)
        заполненные = таблица[маска_заполненных]
        весомых = заполненные["Серьёзность"].astype(str).str.strip().str.lower().isin(ВЕСОМЫЕ).sum()
        коротких = (заполненные["Предложение"].astype(str).str.len() < 20).sum()
        строки_отчёта.append("| Таблица замечаний | записей %d, заполненных %d, весомых %d |"
                             % (len(таблица), len(заполненные), весомых))
        if len(заполненные) < 8:
            ошибки.append("полностью заполненных замечаний %d, требуется не менее 8" % len(заполненные))
        if весомых < 3:
            ошибки.append("замечаний blocker/major %d, требуется не менее 3" % весомых)
        if коротких:
            ошибки.append("замечаний без содержательного предложения: %d" % коротких)

# --- 2. исправленный код ---
def запустить_java(имя_файла):
    if not os.path.exists(имя_файла):
        return False, "файл отсутствует"
    завершение = subprocess.run(["java", имя_файла], capture_output=True, text=True, timeout=180)
    вывод = завершение.stdout + завершение.stderr
    значимые = [с for с in вывод.splitlines() if с.strip() and not с.startswith("Picked up")]
    if завершение.returncode != 0:
        return False, "аварийное завершение: " + (значимые[0][:90] if значимые else "без сообщения")
    if "[FAIL]" in вывод:
        return False, "не пройдено проверок: %d" % вывод.count("[FAIL]")
    успешных = вывод.count("[OK]")
    if успешных < 5:
        return False, "самопроверок в файле только %d, требуется не менее 5" % успешных
    return True, "самопроверок пройдено: %d" % успешных

успех_кода, пояснение_кода = запустить_java("ShiftPlanFixed.java")
строки_отчёта.append("| `ShiftPlanFixed.java` | %s — %s |"
                     % ("пройдено" if успех_кода else "НЕ ПРОЙДЕНО", пояснение_кода))
if not успех_кода:
    ошибки.append("ShiftPlanFixed.java: " + пояснение_кода)

исходный_текст = open("ShiftPlanFixed.java", encoding="utf-8").read() if os.path.exists("ShiftPlanFixed.java") else ""
if исходный_текст and "==" in исходный_текст and re.search(r'==\s*"', исходный_текст):
    ошибки.append("в ShiftPlanFixed.java осталось сравнение строки через ==")

# --- 3. ревью на GitHub ---
ссылка_ревью = globals().get("ССЫЛКА_НА_МОЁ_РЕВЬЮ", "")
ссылка_запроса = globals().get("ССЫЛКА_НА_МОЙ_PR", "")
оставлено = int(globals().get("ЗАМЕЧАНИЙ_ОСТАВЛЕНО", 0) or 0)
получено = int(globals().get("ЗАМЕЧАНИЙ_ПОЛУЧЕНО", 0) or 0)
принято = int(globals().get("ЗАМЕЧАНИЙ_ПРИНЯТО", 0) or 0)

строки_отчёта += [
    "| Ревью на чужой запрос | %s |" % (ссылка_ревью or "НЕ УКАЗАНО"),
    "| Свой запрос | %s |" % (ссылка_запроса or "НЕ УКАЗАН"),
    "| Замечаний оставлено / получено / принято | %d / %d / %d |" % (оставлено, получено, принято),
    "| Таблица прослеживаемости | %s |" % ("заполнена" if ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА else "НЕ ЗАПОЛНЕНА"),
    "| Вопросы и выводы | %s |" % ("заполнены" if ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ else "НЕ ЗАПОЛНЕНЫ"),
]

for подпись, ссылка in [("ревью", ссылка_ревью), ("своего запроса", ссылка_запроса)]:
    if not ОБРАЗЕЦ_ССЫЛКИ.search(str(ссылка)):
        ошибки.append("ссылка %s не похожа на адрес pull request" % подпись)
if оставлено < 5:
    ошибки.append("оставлено замечаний %d, требуется не менее 5" % оставлено)
if получено and принято > получено:
    ошибки.append("принято замечаний больше, чем получено")
if not ФИО_исполнителя.strip() or not Группа_исполнителя.strip():
    ошибки.append("не указаны ФИО и группа")
if not ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА:
    ошибки.append("не заполнена таблица прослеживаемости")
if not ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ:
    ошибки.append("не заполнены контрольные вопросы и выводы")

print("\n".join(строки_отчёта))
print("\nСкопируйте отчёт выше в описание pull request.\n")

assert not ошибки, "Работа не готова к сдаче: " + "; ".join(ошибки)
print("Самопроверка пройдена: работа готова к сдаче.")

### Отчёт самопроверки «Code review»

| Параметр | Значение |
|---|---|
| Исполнитель | Козлов (GitHub: zakharenk0), группа GI-41 (уточнить номер группы) |
| Дата отчёта (UTC) | 2026-11-23 17:07 |
| Таблица замечаний | записей 13, заполненных 13, весомых 11 |
| `ShiftPlanFixed.java` | пройдено — самопроверок пройдено: 11 |
| Ревью на чужой запрос | https://github.com/VeronikaZmushko/Lab8_Code_Review/pull/1#pullrequestreview-6025854716 |
| Свой запрос | https://github.com/vgkozlov/stations-gi41/pull/74 |
| Замечаний оставлено / получено / принято | 7 / 0 / 0 |
| Таблица прослеживаемости | заполнена |
| Вопросы и выводы | заполнены |

Скопируйте отчёт выше в описание pull request.

Самопроверка пройдена: работа готова к сдаче.
